In [ ]:
import json
import numpy as np
import pandas as pd

In [ ]:
path = "atp_results_gemma-2-9b-it_20251201_093733.json"

with open(path, "r") as f:
    data = json.load(f)

In [ ]:
data["results"][1]['variations'][2]["reason"]

In [ ]:
# atp_analysisにerrorが含まれているかをチェック
error_count = 0
error_details = []

for idx, result in enumerate(data["results"]):
    question_id = result.get("question_id", idx)
    
    for var_idx, variation in enumerate(result.get("variations", [])):
        # baseテンプレート(template_type=="")はスキップ
        template_type = variation.get("template_type", "")
        if template_type == "":
            continue
        
        # sycophancy_flag==0の場合もスキップ
        sycophancy_flag = variation.get("sycophancy_flag", 0)
        if sycophancy_flag == 0:
            continue
            
        atp_analysis = variation.get("atp_analysis")
        
        # atp_analysisが存在しない場合
        if atp_analysis is None:
            error_count += 1
            error_details.append({
                "question_id": question_id,
                "variation_index": var_idx,
                "error_type": "missing_atp_analysis",
                "template": variation.get("template_type", "unknown")
            })
        # atp_analysisにerrorキーが含まれている場合
        elif isinstance(atp_analysis, dict) and "error" in atp_analysis:
            error_count += 1
            error_details.append({
                "question_id": question_id,
                "variation_index": var_idx,
                "error_type": "error_in_atp_analysis",
                "error_message": atp_analysis.get("error"),
                "template": variation.get("template_type", "unknown")
            })

print(f"Total errors found: {error_count}")
print(f"\nError details:")
for error in error_details:
    print(f"  Question ID: {error['question_id']}, Variation: {error['variation_index']}, "
          f"Template: {error.get('template')}, Type: {error['error_type']}")
    if "error_message" in error:
        print(f"    Error message: {error['error_message']}")